In [0]:
%pip install databricks-feature-engineering
dbutils.library.restartPython()

In [0]:
import logging
import pickle

import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup

In [0]:
logging.getLogger("mlflow").setLevel(logging.ERROR)
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

In [0]:
class TestingModel:

    def __init__(self,spark,feature_client,feature_table,output_table,model_name):

        self.spark = spark
        self.feature_client = feature_client
        self.feature_table = feature_table
        self.output_table = output_table
        self.model_name = model_name
        self.model_version = None
        self.run_id = None
        logger.info( "Testing Model initialized")
    
    def create_training_dataset(self):

        logger.info("Fetching dataset from Feature Store")
        training_set = self.feature_client.create_training_set(df=self.spark.table(self.output_table),
            feature_lookups=[FeatureLookup(
                    table_name=self.feature_table,
                    lookup_key="primary_key")],
            label="total_demand",
            exclude_columns=["primary_key"])
        df = training_set.load_df()
        logger.info("Feature Store dataset loaded successfully")
        return df

    def get_test_data(self):

        logger.info("Creating test dataset")
        df = self.create_training_dataset()
        df = df.orderBy("transaction_date")
        df = df.drop("transaction_date")
        df = df.toPandas()
        train_validate_df,test_df = train_test_split(df,test_size=0.20,shuffle=False)
        train_df,validate_df = train_test_split(train_validate_df,test_size=0.20,shuffle=False)
        X_test = test_df.drop(columns=["total_demand"])
        y_test = test_df["total_demand"]
        logger.info("Train shape: %s",train_df.shape)
        logger.info("Validation shape: %s",validate_df.shape)
        logger.info("Test shape: %s",test_df.shape)
        logger.info(
            "X_test shape: %s | y_test shape: %s",
            X_test.shape,
            y_test.shape
        )
        return X_test,y_test
    
    def load_model(self):

        logger.info("Loading registered model")

        mlflow.set_registry_uri("databricks-uc")

        client = MlflowClient()

        versions = client.search_model_versions(f"name = '{self.model_name}'")

        if not versions:
            raise ValueError(f"No registered versions found for {self.model_name}")

        registered_model = max(versions,key=lambda x: int(x.version))

        self.model_version = registered_model.version
        self.run_id = registered_model.run_id

        model_uri = f"models:/{self.model_name}/{self.model_version}"

        logger.info(
            "Model URI: %s",
            model_uri
        )

        logger.info(
            "Model version: %s",
            self.model_version
        )

        logger.info(
            "Associated run ID: %s",
            self.run_id
        )

        model = mlflow.sklearn.load_model(model_uri)

        logger.info("Registered model loaded successfully")

        return model
    
    def load_encoder(self,run_id):

        logger.info("Loading encoder from MLflow run")
        encoder_path = mlflow.artifacts.download_artifacts(
            run_id=run_id,
            artifact_path="preprocessing/onehot_encoder.pkl")

        with open(encoder_path,"rb") as file:
            encoder = pickle.load(file)

        logger.info("Encoder loaded successfully")
        return encoder
    
    def encode_test_data(self,X_test,encoder):

        logger.info("Encoding test data")
        category_columns = ["product_name","destination_city"]
        numeric_columns = ["avg_unit_price","total_inventory","transaction_count","demand_1","demand_7","rolling_7_day_avg","month","day_of_week"]
        encoded_data = encoder.transform(X_test[category_columns])
        encoded_columns = encoder.get_feature_names_out(category_columns)
        encoded_df = pd.DataFrame(
            encoded_data,
            columns=encoded_columns,
            index=X_test.index
        )

        X_test_final = pd.concat(
            [
                X_test[numeric_columns],
                encoded_df
            ],
            axis=1
        )

        logger.info(
            "Encoded test data shape: %s",
            X_test_final.shape
        )

        return X_test_final
    
    def test_model(self,X_test,y_test):

        logger.info("Testing registered model")
        model = self.load_model()
        encoder = self.load_encoder(self.run_id)

        X_test_final = self.encode_test_data(X_test,encoder)
        predictions = model.predict(X_test_final)

        test_r2 = r2_score(y_test,predictions)

        logger.info(
            "Test R2: %.4f",
            test_r2
        )

        return test_r2,predictions
    
    def validate_challenger(self,test_r2):

        client = MlflowClient()

        client.set_model_version_tag(
            name=self.model_name,
            version=self.model_version,
            key="test_r2",
            value=str(test_r2)
        )

        if test_r2 > 0.90:

            client.set_model_version_tag(
                name=self.model_name,
                version=self.model_version,
                key="test_r2_status",
                value="cleared"
            )

            client.set_registered_model_alias(
                name=self.model_name,
                alias="Challenger",
                version=self.model_version
            )

            logger.info(
                "Model version %s passed test R2 and assigned Challenger alias",
                self.model_version
            )

            return True

        client.set_model_version_tag(
            name=self.model_name,
            version=self.model_version,
            key="test_r2_status",
            value="not_cleared"
        )

        logger.info(
            "Model version %s did not pass test R2 threshold",
            self.model_version
        )
        return False
    
    def run(self):

        logger.info(
            "Testing pipeline started"
        )

        X_test,y_test = self.get_test_data()

        test_r2,predictions = self.test_model(X_test,y_test)
        flag = self.validate_challenger(test_r2)

        if flag:
            logger.info("Conditions cleared")
        else:
            logger.info("Conditions not cleared")
        

        logger.info(
            "Testing pipeline completed"
        )

        return flag

In [0]:
feature_client = FeatureEngineeringClient()

tst_model = TestingModel(
    spark=spark,
    feature_client=feature_client,
    feature_table="oag.gold.demand_input_features",
    output_table="oag.gold.demand_output",
    model_name="oag.ml.demand_prediction_model")
tst_model.run()